In [ ]:
!nvidia-smi

# Qwen 2.5 7B Instruct: comprehensive selective prediction + Instruct mechanistic

Model: Qwen/Qwen2.5-7B-Instruct | [HF](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)\
GPU: H100 SXM 80GB | 3 seeds | Concat probe | Logit lens | Head-level L0 | Instruct ablation L0-14 | Date: 2026-04-07

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

In [ ]:
!pip install transformers datasets scipy huggingface_hub -q

try:
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("Not on Colab or no HF_TOKEN secret")

In [ ]:
# Pre-download models (uses HF_TOKEN from env)
!hf download Qwen/Qwen2.5-7B-Instruct


In [ ]:
import gc
import json
import re
import string

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import mannwhitneyu, pearsonr, rankdata
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
def load_wikitext(split='test', max_docs=None):
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split=split)
    docs, current = [], []
    for row in ds:
        text = row['text']
        if text.strip() == '' and current:
            docs.append('\n'.join(current)); current = []
            if max_docs and len(docs) >= max_docs: break
        elif text.strip(): current.append(text)
    if current: docs.append('\n'.join(current))
    return docs

def load_triviaqa(split='validation', max_questions=2000):
    from datasets import load_dataset
    ds = load_dataset('trivia_qa', 'rc.nocontext', split=split)
    questions = []
    for row in ds:
        if len(questions) >= max_questions: break
        answers = row['answer']['aliases'] + [row['answer']['value']]
        seen, unique = set(), []
        for a in answers:
            norm = a.strip().lower()
            if norm not in seen: seen.add(norm); unique.append(a.strip())
        questions.append({'question': row['question'], 'answers': unique})
    return questions

def compute_loss_residuals(losses, max_softmax, activation_norm):
    X = np.column_stack([max_softmax, activation_norm, np.ones(len(losses))])
    beta, _, _, _ = np.linalg.lstsq(X, losses, rcond=None)
    return losses - X @ beta

def collect_layer_data(model, tokenizer, docs, layer, device, max_tokens=200000, max_length=512):
    model.eval()
    all_acts, all_losses, all_softmax, all_norms = [], [], [], []
    total = 0
    for doc in docs:
        if total >= max_tokens: break
        if not doc.strip(): continue
        tokens = tokenizer(doc, return_tensors='pt', truncation=True, max_length=max_length)
        input_ids = tokens['input_ids'].to(device)
        if input_ids.size(1) < 2: continue
        with torch.inference_mode():
            outputs = model(input_ids, output_hidden_states=True)
        h = outputs.hidden_states[layer + 1][0, :-1, :].cpu()
        logits = outputs.logits[0, :-1, :]
        labels = input_ids[0, 1:]
        losses = F.cross_entropy(logits, labels, reduction='none').cpu()
        sm = F.softmax(logits, dim=-1).max(dim=-1).values.cpu()
        norms = h.norm(dim=-1)
        all_acts.append(h); all_losses.append(losses); all_softmax.append(sm); all_norms.append(norms)
        total += h.size(0)
    print(f'    {total} positions from {len(all_acts)} documents')
    return {
        'activations': torch.cat(all_acts).float(),
        'losses': torch.cat(all_losses).float().numpy(),
        'max_softmax': torch.cat(all_softmax).float().numpy(),
        'activation_norm': torch.cat(all_norms).float().numpy(),
    }

def train_linear_binary(train_data, seed=42, epochs=20, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    acts = train_data['activations']
    residuals = compute_loss_residuals(
        train_data['losses'], train_data['max_softmax'], train_data['activation_norm']
    )
    targets = torch.from_numpy((residuals > 0).astype(np.float32))
    head = torch.nn.Linear(acts.size(1), 1)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(acts, targets)
    dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
    head.train()
    for _ in range(epochs):
        for bx, by in dl:
            loss = F.binary_cross_entropy_with_logits(head(bx).squeeze(-1), by)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return head

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())

def exact_match(prediction, references):
    norm_pred = normalize_answer(prediction)
    return any(normalize_answer(ref) == norm_pred for ref in references)

def format_qa_prompt(question, tokenizer):
    if hasattr(tokenizer, 'apply_chat_template'):
        messages = [{'role': 'user', 'content': f'Answer the following question in as few words as possible.\n\nQuestion: {question}\nAnswer:'}]
        try: return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception: pass
    return f'Question: {question}\nAnswer:'

def _get_layer_modules(model, layer_idx):
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        block = model.model.layers[layer_idx]
        return block.self_attn, block.mlp
    if hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        block = model.transformer.h[layer_idx]
        return block.attn, block.mlp
    raise ValueError(f'Unsupported: {type(model).__name__}')

## Load model

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
PROBE_LAYERS = [0, 7, 14]
PEAK_LAYER = 14
MAX_NEW_TOKENS = 64
SEEDS = [42, 43, 44]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16,
    attn_implementation='sdpa'
).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
N_HEADS = model.config.num_attention_heads
HEAD_DIM = HIDDEN_DIM // N_HEADS
N_PARAMS = sum(p.numel() for p in model.parameters()) / 1e9
print(f'{N_PARAMS:.1f}B params, {N_LAYERS} layers, {HIDDEN_DIM} dim, {N_HEADS} heads ({HEAD_DIM} dim/head)')

In [ ]:
questions = load_triviaqa('validation', max_questions=2000)
print(f'{len(questions)} questions')

## Phase 1: Generate all answers (multi-layer)

In [ ]:
generation_cache = []

for i, q in enumerate(questions):
    prompt = format_qa_prompt(q['question'], tokenizer)
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids'].cuda()
    prompt_len = input_ids.size(1)
    per_layer_h = {L: [] for L in PROBE_LAYERS}
    confidences, entropies, logit_margins, gen_token_ids = [], [], [], []
    past_key_values = None

    with torch.inference_mode():
        for step in range(MAX_NEW_TOKENS):
            chunk = input_ids if past_key_values is None else input_ids[:, -1:]
            out = model(chunk, past_key_values=past_key_values,
                        output_hidden_states=True, use_cache=True)
            past_key_values = out.past_key_values

            for L in PROBE_LAYERS:
                per_layer_h[L].append(out.hidden_states[L + 1][0, -1, :].cpu().float())

            logits = out.logits[0, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = logits.argmax(dim=-1, keepdim=True)
            gen_token_ids.append(next_token.item())

            confidences.append(probs.max().item())
            entropies.append(-(probs * torch.log(probs + 1e-10)).sum().item())
            top2 = probs.topk(2).values
            logit_margins.append((top2[0] - top2[1]).item())

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            if next_token.item() == tokenizer.eos_token_id: break
            if '\n' in tokenizer.decode(next_token.item()) and step > 0: break

    answer_text = tokenizer.decode(gen_token_ids, skip_special_tokens=True).strip()
    generation_cache.append({
        'question': q['question'],
        'answers': q['answers'],
        'answer_text': answer_text,
        'correct': exact_match(answer_text, q['answers']),
        'hidden_states': {L: torch.stack(per_layer_h[L]) for L in PROBE_LAYERS},
        'confidences': np.array(confidences),
        'entropies': np.array(entropies),
        'logit_margins': np.array(logit_margins),
        'token_ids': gen_token_ids,
    })

    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(questions)}  EM={np.mean([g["correct"] for g in generation_cache]):.3f}')

base_acc = np.mean([g['correct'] for g in generation_cache])
print(f'\nBase EM: {base_acc:.3f} ({sum(g["correct"] for g in generation_cache)}/{len(generation_cache)})')

## Phase 2: Selective prediction (multi-layer + concatenated probe)

In [ ]:
wiki_train = load_wikitext('train', max_docs=2000)
MAX_TRAIN = max(150 * HIDDEN_DIM, 200000)

observers = {}
wiki_train_cache = {}
for L in PROBE_LAYERS:
    print(f'Layer {L}:')
    td = collect_layer_data(model, tokenizer, wiki_train, L, 'cuda', MAX_TRAIN)
    wiki_train_cache[L] = td
    observers[L] = {}
    for seed in SEEDS:
        observers[L][seed] = train_linear_binary(td, seed=seed)
        observers[L][seed].eval()

# Concatenated probe: [layer_0; layer_14]
print('\nConcatenated [L0;L14] probe:')
concat_train = {
    'activations': torch.cat([wiki_train_cache[0]['activations'],
                              wiki_train_cache[14]['activations']], dim=1),
    'losses': wiki_train_cache[0]['losses'],
    'max_softmax': wiki_train_cache[0]['max_softmax'],
    'activation_norm': wiki_train_cache[0]['activation_norm'],
}
n_min = min(len(concat_train['losses']), concat_train['activations'].size(0))
concat_train = {k: v[:n_min] if hasattr(v, '__getitem__') else v for k, v in concat_train.items()}
observers['concat'] = {}
for seed in SEEDS:
    observers['concat'][seed] = train_linear_binary(concat_train, seed=seed)
    observers['concat'][seed].eval()
print(f'  Input dim: {concat_train["activations"].size(1)}')

del wiki_train_cache, concat_train; gc.collect(); torch.cuda.empty_cache()

In [ ]:
coverage_levels = list(np.arange(0.5, 1.01, 0.05))
correct = np.array([g['correct'] for g in generation_cache])
n = len(correct)

def auacc(order):
    accs = [float(correct[order[:max(1, int(n * c))]].mean()) for c in coverage_levels]
    return float(np.trapz(accs, coverage_levels))

conf_mean = np.array([g['confidences'].mean() for g in generation_cache])
conf_min = np.array([g['confidences'].min() for g in generation_cache])
ent_mean = np.array([g['entropies'].mean() for g in generation_cache])
margin_mean = np.array([g['logit_margins'].mean() for g in generation_cache])

print('OUTPUT BASELINES')
baselines = {
    'confidence_mean': auacc(np.argsort(-conf_mean)),
    'confidence_min': auacc(np.argsort(-conf_min)),
    'entropy_mean': auacc(np.argsort(ent_mean)),
    'margin_mean': auacc(np.argsort(-margin_mean)),
}
for k, v in baselines.items():
    print(f'  {k:<30} AUACC={v:.4f}')

In [ ]:
print('OBSERVER STRATEGIES\n')
all_results = {}

probe_keys = list(PROBE_LAYERS) + ['concat']

for L in probe_keys:
    print(f'--- {"L" + str(L) if isinstance(L, int) else L} ---')
    layer_results = {}

    for agg in ['mean', 'max', 'first', 'flag_frac']:
        seed_auaccs = []
        for seed in SEEDS:
            head = observers[L][seed]
            per_q = []
            with torch.inference_mode():
                for g in generation_cache:
                    if L == 'concat':
                        h = torch.cat([g['hidden_states'][0], g['hidden_states'][14]], dim=1)
                    else:
                        h = g['hidden_states'][L]
                    s = head(h).squeeze(-1).numpy()
                    if agg == 'mean': per_q.append(s.mean())
                    elif agg == 'max': per_q.append(s.max())
                    elif agg == 'first': per_q.append(s[0])
                    elif agg == 'flag_frac':
                        if not hasattr(head, '_thresh'):
                            all_s = []
                            for g2 in generation_cache:
                                h2 = torch.cat([g2['hidden_states'][0], g2['hidden_states'][14]], dim=1) if L == 'concat' else g2['hidden_states'][L]
                                all_s.append(head(h2).squeeze(-1).numpy())
                            head._thresh = float(np.percentile(np.concatenate(all_s), 90))
                        per_q.append((s > head._thresh).mean())
            seed_auaccs.append(auacc(np.argsort(np.array(per_q))))

        m = np.mean(seed_auaccs)
        key = f'{"L" + str(L) if isinstance(L, int) else L}_{agg}'
        layer_results[agg] = {'auacc_mean': m, 'per_seed': seed_auaccs}
        print(f'  {key:<30} AUACC={m:.4f}  seeds={[round(x,4) for x in seed_auaccs]}')

    all_results[str(L)] = layer_results
    print()

In [ ]:
# Quadrant analysis
conf_median = np.median(conf_mean)
confident = conf_mean >= conf_median
wrong_confident = ~correct & confident
correct_confident = correct & confident

print(f'Confident: {confident.sum()}, wrong+confident: {wrong_confident.sum()}, correct+confident: {correct_confident.sum()}\n')
print(f'{"Strategy":<30} {"AUROC":>8} {"p":>10}')
print('-' * 50)

quadrant_results = {}

for L in probe_keys:
    head = observers[L][42]
    with torch.inference_mode():
        all_scores = []
        for g in generation_cache:
            h = torch.cat([g['hidden_states'][0], g['hidden_states'][14]], dim=1) if L == 'concat' else g['hidden_states'][L]
            all_scores.append(head(h).squeeze(-1).numpy())

    for agg_name, agg_fn in [('mean', np.mean), ('max', np.max), ('first', lambda s: s[0]),
                              ('flag_frac', lambda s: (s > head._thresh).mean() if hasattr(head, '_thresh') else 0)]:
        per_q = np.array([agg_fn(s) for s in all_scores])
        wc, cc = per_q[wrong_confident], per_q[correct_confident]
        if len(wc) > 5 and len(cc) > 5:
            U, p = mannwhitneyu(wc, cc, alternative='greater')
            a = U / (len(wc) * len(cc))
            key = f'{"L" + str(L) if isinstance(L, int) else L}_{agg_name}'
            quadrant_results[key] = {'auroc': round(a, 4), 'p': round(p, 6)}
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f'  {key:<28} {a:>8.3f} {p:>10.4f} {sig}')

for name, scores in [('entropy', ent_mean), ('neg_confidence', -conf_mean), ('neg_margin', -margin_mean)]:
    wc, cc = scores[wrong_confident], scores[correct_confident]
    if len(wc) > 5 and len(cc) > 5:
        U, p = mannwhitneyu(wc, cc, alternative='greater')
        a = U / (len(wc) * len(cc))
        quadrant_results[name] = {'auroc': round(a, 4), 'p': round(p, 6)}
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f'  {name:<28} {a:>8.3f} {p:>10.4f} {sig}')

## Phase 3: Logit lens at layer 0

In [ ]:
unembed = model.lm_head.weight.float().cpu()  # [vocab, hidden]

obs_head_L0 = observers[0][42]
obs_head_L0.eval()

agree_correct, agree_wrong = [], []
obs_scores_agree, obs_scores_disagree = [], []

for g in generation_cache:
    h0 = g['hidden_states'][0]  # [n_tokens, hidden]
    with torch.inference_mode():
        obs_s = obs_head_L0(h0).squeeze(-1).numpy()
        logit_lens = h0 @ unembed.T  # [n_tokens, vocab]
        l0_preds = logit_lens.argmax(dim=-1).numpy()  # what layer 0 predicts

    actual_tokens = np.array(g['token_ids'][:len(l0_preds)])
    agrees = (l0_preds == actual_tokens)

    for tok_idx in range(len(obs_s)):
        if agrees[tok_idx]:
            obs_scores_agree.append(obs_s[tok_idx])
        else:
            obs_scores_disagree.append(obs_s[tok_idx])

    agree_rate = agrees.mean()
    if g['correct']:
        agree_correct.append(agree_rate)
    else:
        agree_wrong.append(agree_rate)

obs_agree = np.array(obs_scores_agree)
obs_disagree = np.array(obs_scores_disagree)

print('LOGIT LENS AT LAYER 0')
print(f'Tokens where L0 agrees with final output: {len(obs_agree)}')
print(f'Tokens where L0 disagrees: {len(obs_disagree)}')
print(f'\nMean observer score (L0 agrees):    {obs_agree.mean():+.4f}')
print(f'Mean observer score (L0 disagrees):  {obs_disagree.mean():+.4f}')

U, p = mannwhitneyu(obs_disagree, obs_agree, alternative='greater')
auroc_lens = U / (len(obs_disagree) * len(obs_agree))
print(f'AUROC (disagree vs agree): {auroc_lens:.3f}  p={p:.6f}')

print(f'\nL0 agreement rate on correct answers: {np.mean(agree_correct):.3f}')
print(f'L0 agreement rate on wrong answers:   {np.mean(agree_wrong):.3f}')

if len(agree_correct) > 5 and len(agree_wrong) > 5:
    U2, p2 = mannwhitneyu(agree_wrong, agree_correct, alternative='less')
    auroc_qa = U2 / (len(agree_wrong) * len(agree_correct))
    print(f'AUROC (agreement rate separating correct vs wrong questions): {1-auroc_qa:.3f}  p={p2:.6f}')

logit_lens_results = {
    'n_agree': len(obs_agree), 'n_disagree': len(obs_disagree),
    'mean_obs_agree': float(obs_agree.mean()), 'mean_obs_disagree': float(obs_disagree.mean()),
    'auroc_token_level': round(auroc_lens, 4), 'p_token_level': round(p, 6),
    'mean_agree_rate_correct': float(np.mean(agree_correct)),
    'mean_agree_rate_wrong': float(np.mean(agree_wrong)),
}
del unembed; gc.collect()

## Phase 4: Token-level examples

In [ ]:
obs_head = observers[0][42]
obs_head.eval()

examples = []
correct_qs = [i for i, g in enumerate(generation_cache) if g['correct']]
wrong_conf_qs = [i for i, g in enumerate(generation_cache) if not g['correct'] and conf_mean[i] >= conf_median]
wrong_unconf_qs = [i for i, g in enumerate(generation_cache) if not g['correct'] and conf_mean[i] < conf_median]

np.random.seed(42)
sample_ids = (
    list(np.random.choice(correct_qs, min(10, len(correct_qs)), replace=False))
    + list(np.random.choice(wrong_conf_qs, min(10, len(wrong_conf_qs)), replace=False))
    + list(np.random.choice(wrong_unconf_qs, min(10, len(wrong_unconf_qs)), replace=False))
)

for idx in sample_ids:
    g = generation_cache[idx]
    with torch.inference_mode():
        scores_L0 = obs_head(g['hidden_states'][0]).squeeze(-1).numpy()
    tokens = [tokenizer.decode([tid]) for tid in g['token_ids'][:len(scores_L0)]]
    examples.append({
        'question': g['question'],
        'answer': g['answer_text'],
        'references': g['answers'][:3],
        'correct': g['correct'],
        'mean_confidence': float(g['confidences'].mean()),
        'category': 'correct' if g['correct'] else ('wrong_confident' if conf_mean[idx] >= conf_median else 'wrong_uncertain'),
        'tokens': tokens,
        'observer_scores_L0': [round(float(s), 4) for s in scores_L0],
        'confidences': [round(float(c), 4) for c in g['confidences'][:len(scores_L0)]],
    })

print(f'{len(examples)} examples saved')
for ex in examples[:5]:
    print(f'  [{ex["category"]}] Q: {ex["question"][:60]}... A: {ex["answer"][:30]}')
    print(f'    obs_scores: {ex["observer_scores_L0"][:8]}...')

## Phase 5: Mechanistic ablation (Instruct, peak layer 14)

In [ ]:
wiki_test = load_wikitext('test', max_docs=500)
EVAL_BUDGET = 15000
eval_docs = wiki_test[:200]

mech_train = collect_layer_data(model, tokenizer, load_wikitext('train', max_docs=2000), PEAK_LAYER, 'cuda', MAX_TRAIN)
mech_observer = train_linear_binary(mech_train, seed=42)
mech_observer.eval()
del mech_train; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Component means
accumulators = {}
def make_acc_hook(key):
    def hook_fn(module, inp, out):
        val = out[0] if isinstance(out, tuple) else out
        if key not in accumulators: accumulators[key] = []
        accumulators[key].append(val.detach().cpu())
        return out
    return hook_fn

handles = []
for layer in range(PEAK_LAYER + 1):
    attn, mlp = _get_layer_modules(model, layer)
    handles.append(attn.register_forward_hook(make_acc_hook((layer, 'attn'))))
    handles.append(mlp.register_forward_hook(make_acc_hook((layer, 'mlp'))))

total = 0
with torch.inference_mode():
    for doc in eval_docs:
        if total >= EVAL_BUDGET: break
        if not doc.strip(): continue
        ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].cuda()
        if ids.size(1) < 2: continue
        model(ids)
        total += ids.size(1) - 1

for h in handles: h.remove()
component_means = {}
for key, tensors in accumulators.items():
    component_means[key] = torch.cat(tensors, dim=1).mean(dim=1, keepdim=True)
del accumulators; gc.collect()
print(f'{len(component_means)} components, {total} positions')

In [ ]:
def collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens):
    all_obs, all_loss, all_sm = [], [], []
    total = 0
    model.eval(); observer.eval()
    with torch.inference_mode():
        for doc in docs:
            if total >= max_tokens: break
            if not doc.strip(): continue
            ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].to(device)
            if ids.size(1) < 2: continue
            out = model(ids, output_hidden_states=True)
            h = out.hidden_states[peak_layer + 1][0, :-1, :].cpu().float()
            logits = out.logits[0, :-1, :]
            losses = F.cross_entropy(logits, ids[0, 1:], reduction='none').cpu().float().numpy()
            sm = F.softmax(logits, dim=-1).max(dim=-1).values.cpu().float().numpy()
            all_obs.append(observer(h).squeeze(-1).numpy())
            all_loss.append(losses); all_sm.append(sm)
            total += h.size(0)
    return {
        'obs_scores': np.concatenate(all_obs)[:max_tokens],
        'losses': np.concatenate(all_loss)[:max_tokens],
        'max_softmax': np.concatenate(all_sm)[:max_tokens],
        'n': min(total, max_tokens),
    }

baseline = collect_baseline(model, tokenizer, mech_observer, PEAK_LAYER, eval_docs, 'cuda', EVAL_BUDGET)
print(f'Baseline: {baseline["n"]} positions')

In [ ]:
def residualized_delta(base_obs, patched_obs, base_sm, patched_sm):
    sm_d = patched_sm - base_sm; obs_d = patched_obs - base_obs
    if np.std(sm_d) > 1e-8:
        beta = np.cov(obs_d, sm_d)[0, 1] / np.var(sm_d)
        return float((obs_d - beta * sm_d).mean())
    return float(obs_d.mean())

def mean_ablate(model, tokenizer, observer, peak_layer, docs, device, max_tokens, target_layer, component, means):
    mv = means[(target_layer, component)]
    attn, mlp = _get_layer_modules(model, target_layer)
    target = attn if component == 'attn' else mlp
    def hook_fn(module, inp, out):
        m = mv.to(out[0].device if isinstance(out, tuple) else out.device)
        expanded = m.expand_as(out[0] if isinstance(out, tuple) else out)
        if isinstance(out, tuple): return (expanded,) + out[1:]
        return expanded
    handle = target.register_forward_hook(hook_fn)
    result = collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens)
    handle.remove()
    return result

print(f'{"Layer":<8} {"Comp":<6} {"Obs resid":>12} {"Loss":>10}')
print('-' * 38)
ablation_results = {}
for layer in range(PEAK_LAYER + 1):
    layer_r = {}
    for comp in ['attn', 'mlp']:
        p = mean_ablate(model, tokenizer, mech_observer, PEAK_LAYER, eval_docs, 'cuda',
                        EVAL_BUDGET, layer, comp, component_means)
        obs_r = residualized_delta(baseline['obs_scores'], p['obs_scores'], baseline['max_softmax'], p['max_softmax'])
        loss_d = float(p['losses'].mean() - baseline['losses'].mean())
        layer_r[comp] = {'obs_resid_delta': obs_r, 'loss_delta': loss_d}
        print(f'{layer:<8} {comp:<6} {obs_r:>+12.4f} {loss_d:>+10.4f}')
    ablation_results[layer] = layer_r

In [ ]:
# Composition
def mean_ablate_group(model, tokenizer, observer, peak_layer, docs, device, max_tokens, components, means):
    handles = []
    for layer, comp in components:
        mv = means[(layer, comp)]
        attn, mlp = _get_layer_modules(model, layer)
        target = attn if comp == 'attn' else mlp
        def make_hook(m):
            def hook_fn(module, inp, out):
                mv = m.to(out[0].device if isinstance(out, tuple) else out.device)
                expanded = mv.expand_as(out[0] if isinstance(out, tuple) else out)
                if isinstance(out, tuple): return (expanded,) + out[1:]
                return expanded
            return hook_fn
        handles.append(target.register_forward_hook(make_hook(mv)))
    result = collect_baseline(model, tokenizer, observer, peak_layer, docs, device, max_tokens)
    for h in handles: h.remove()
    return result

attn_ranked = sorted([(l, ablation_results[l]['attn']['obs_resid_delta']) for l in range(PEAK_LAYER + 1)],
                     key=lambda x: abs(x[1]), reverse=True)
mlp_ranked = sorted([(l, ablation_results[l]['mlp']['obs_resid_delta']) for l in range(PEAK_LAYER + 1)],
                    key=lambda x: abs(x[1]), reverse=True)

top2_attn = sorted([attn_ranked[0][0], attn_ranked[1][0]])
top4_attn = sorted([r[0] for r in attn_ranked[:4]])
top2_mlp = sorted([mlp_ranked[0][0], mlp_ranked[1][0]])

groups = {
    f'attn_{top2_attn[0]}_{top2_attn[1]}': [(l, 'attn') for l in top2_attn],
    'attn_top4': [(l, 'attn') for l in top4_attn],
    f'mlp_{top2_mlp[0]}_{top2_mlp[1]}': [(l, 'mlp') for l in top2_mlp],
    'all_top': [(l, 'attn') for l in top4_attn] + [(l, 'mlp') for l in top2_mlp],
}

composition_results = {}
for gname, components in groups.items():
    p = mean_ablate_group(model, tokenizer, mech_observer, PEAK_LAYER, eval_docs, 'cuda',
                          EVAL_BUDGET, components, component_means)
    obs_r = residualized_delta(baseline['obs_scores'], p['obs_scores'], baseline['max_softmax'], p['max_softmax'])
    expected = sum(ablation_results[l][c]['obs_resid_delta'] for l, c in components)
    composition_results[gname] = {
        'obs_resid_delta': obs_r, 'expected_additive': expected,
        'interaction': obs_r - expected, 'components': [[l, c] for l, c in components],
    }
    tag = 'additive' if abs(obs_r - expected) < 0.01 else f'interaction={obs_r - expected:+.4f}'
    print(f'{gname:<22}: obs_resid={obs_r:+.4f}  expected={expected:+.4f}  ({tag})')

## Phase 6: Head-level decomposition at layer 0

In [ ]:
attn_0, _ = _get_layer_modules(model, 0)
o_proj = attn_0.o_proj

oproj_inputs = []
def capture_oproj(module, inp):
    oproj_inputs.append(inp[0].detach().cpu())
    return inp

handle = o_proj.register_forward_pre_hook(capture_oproj)
total = 0
with torch.inference_mode():
    for doc in eval_docs:
        if total >= EVAL_BUDGET: break
        if not doc.strip(): continue
        ids = tokenizer(doc, return_tensors='pt', truncation=True, max_length=512)['input_ids'].cuda()
        if ids.size(1) < 2: continue
        model(ids)
        total += ids.size(1) - 1
handle.remove()

oproj_cat = torch.cat(oproj_inputs, dim=1)
oproj_mean = oproj_cat.mean(dim=1, keepdim=True)  # [1, 1, hidden_dim]
del oproj_inputs, oproj_cat; gc.collect()
print(f'o_proj input mean computed over {total} positions')

# Per-head ablation
head_results = {}
print(f'\n{"Head":<8} {"Obs resid":>12} {"Loss":>10}')
print('-' * 32)

for h_idx in range(N_HEADS):
    start = h_idx * HEAD_DIM
    end = start + HEAD_DIM
    head_mean = oproj_mean[:, :, start:end]

    def make_head_hook(s, e, hm):
        def hook_fn(module, inp):
            x = inp[0].clone()
            x[:, :, s:e] = hm.to(x.device)
            return (x,)
        return hook_fn

    handle = o_proj.register_forward_pre_hook(make_head_hook(start, end, head_mean))
    patched = collect_baseline(model, tokenizer, mech_observer, PEAK_LAYER, eval_docs, 'cuda', EVAL_BUDGET)
    handle.remove()

    obs_r = residualized_delta(baseline['obs_scores'], patched['obs_scores'],
                               baseline['max_softmax'], patched['max_softmax'])
    loss_d = float(patched['losses'].mean() - baseline['losses'].mean())
    head_results[h_idx] = {'obs_resid_delta': obs_r, 'loss_delta': loss_d}
    print(f'{h_idx:<8} {obs_r:>+12.4f} {loss_d:>+10.4f}')

sorted_heads = sorted(head_results.items(), key=lambda x: abs(x[1]['obs_resid_delta']), reverse=True)
print(f'\nTop 5 heads: {[(h, round(v["obs_resid_delta"], 4)) for h, v in sorted_heads[:5]]}')

## Save

In [ ]:
output = {
    'model': MODEL_ID,
    'probe_layers': PROBE_LAYERS,
    'peak_layer': PEAK_LAYER,
    'n_questions': len(questions),
    'base_em_accuracy': float(base_acc),
    'output_baselines': baselines,
    'observer_results': {str(k): v for k, v in all_results.items()},
    'quadrant_analysis': {
        'n_confident': int(confident.sum()),
        'n_wrong_confident': int(wrong_confident.sum()),
        'n_correct_confident': int(correct_confident.sum()),
        'auroc_results': quadrant_results,
    },
    'logit_lens': logit_lens_results,
    'token_examples': examples,
    'instruct_mechanistic': {
        'n_eval': baseline['n'],
        'ablation_results': {str(k): v for k, v in ablation_results.items()},
        'composition_results': composition_results,
        'top_attn_layers': [r[0] for r in attn_ranked[:3]],
        'top_mlp_layers': [r[0] for r in mlp_ranked[:3]],
    },
    'head_level_layer0': {
        'n_heads': N_HEADS,
        'head_dim': HEAD_DIM,
        'per_head': {str(k): v for k, v in head_results.items()},
        'top5': [(h, round(v['obs_resid_delta'], 4)) for h, v in sorted_heads[:5]],
    },
}

with open('comprehensive_v3_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print(json.dumps({k: v for k, v in output.items() if k not in ('token_examples',)}, indent=2)[:3000])

In [ ]:
from google.colab import files
files.download('comprehensive_v3_results.json')